In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plot style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

print('Libraries loaded successfully ✅')

df = pd.read_csv('data/Student Placement Dataset/train.csv')
print(f'Dataset shape: {df.shape}')
df.head()

print('=== Data Types ===')
print(df.dtypes)
print('\n=== Basic Statistics ===')
df.describe(include='all')

print('=== Missing Values ===')
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

target_col = 'PlacementStatus'   # ← update if column name differs

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
counts = df[target_col].value_counts()
axes[0].bar(counts.index, counts.values, color=['#2ecc71', '#e74c3c'], edgecolor='black')
axes[0].set_title('Placement Distribution (Count)', fontsize=14)
axes[0].set_xlabel(target_col)
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=['#2ecc71', '#e74c3c'], startangle=90)
axes[1].set_title('Placement Distribution (%)', fontsize=14)

plt.tight_layout()
plt.savefig('models/placement_distribution.png', bbox_inches='tight')
plt.show()
print(counts)

num_cols = df.select_dtypes(include=np.number).columns.tolist()
if target_col in num_cols:
    num_cols.remove(target_col)

n = len(num_cols)
cols = 3
rows = (n + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(18, rows * 4))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(df[col].dropna(), bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i].set_title(col, fontsize=12)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frequency')

# Hide unused axes
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Numerical Feature Distributions', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

cgpa_col = 'CGPA'   # ← update if column name differs

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# KDE plot
for label, grp in df.groupby(target_col):
    grp[cgpa_col].plot(kind='kde', ax=axes[0], label=label, linewidth=2)
axes[0].set_title('CGPA Distribution by Placement Status')
axes[0].set_xlabel(cgpa_col)
axes[0].legend()

# Box plot
sns.boxplot(data=df, x=target_col, y=cgpa_col, palette=['#2ecc71', '#e74c3c'], ax=axes[1])
axes[1].set_title('CGPA Box Plot by Placement Status')

plt.tight_layout()
plt.savefig('models/cgpa_distribution.png', bbox_inches='tight')
plt.show()

cat_cols = df.select_dtypes(include='object').columns.tolist()
if target_col in cat_cols:
    cat_cols.remove(target_col)

for col in cat_cols:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    
    # Value counts
    vc = df[col].value_counts()
    axes[0].bar(vc.index, vc.values, color='steelblue', edgecolor='black')
    axes[0].set_title(f'{col} - Distribution')
    axes[0].tick_params(axis='x', rotation=30)
    
    # Placement rate per category
    ct = pd.crosstab(df[col], df[target_col], normalize='index') * 100
    ct.plot(kind='bar', ax=axes[1], colormap='RdYlGn', edgecolor='black')
    axes[1].set_title(f'{col} - Placement Rate (%)')
    axes[1].set_ylabel('%')
    axes[1].tick_params(axis='x', rotation=30)
    axes[1].legend(title=target_col)
    
    plt.tight_layout()
    plt.show()

    skill_cols = ['Coding_Skills', 'Communication_Skills', 'Aptitude_Test_Score',
              'Soft_Skills', 'Internships', 'Projects', 'Certifications']

# Keep only columns that exist
skill_cols = [c for c in skill_cols if c in df.columns]

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, col in enumerate(skill_cols):
    sns.violinplot(data=df, x=target_col, y=col,
                   palette=['#2ecc71', '#e74c3c'], ax=axes[i], inner='quartile')
    axes[i].set_title(col, fontsize=11)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Skill Features vs Placement Status', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

# Encode target for correlation
df_enc = df.copy()
df_enc[target_col] = df_enc[target_col].map({'Placed': 1, 'Not Placed': 0})

corr = df_enc[num_cols + [target_col]].corr()

plt.figure(figsize=(14, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, linewidths=0.5,
            annot_kws={'size': 9})
plt.title('Feature Correlation Matrix', fontsize=15)
plt.tight_layout()
plt.savefig('models/correlation_matrix.png', bbox_inches='tight')
plt.show()

# Top correlations with target
target_corr = corr[target_col].drop(target_col).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in target_corr.values]
plt.barh(target_corr.index, target_corr.values, color=colors, edgecolor='black')
plt.axvline(0, color='black', linewidth=0.8)
plt.title(f'Feature Correlations with {target_col}', fontsize=14)
plt.xlabel('Pearson Correlation')
plt.tight_layout()
plt.show()
print(target_corr)

fig, axes = plt.subplots(2, (len(num_cols) + 1) // 2, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.boxplot(data=df, y=col, ax=axes[i], color='steelblue')
    axes[i].set_title(col)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Outlier Detection – Box Plots', fontsize=15)
plt.tight_layout()
plt.show()

df_fe = df_enc.copy()

# Normalise numeric cols to 0-1 for composite scores
for col in num_cols:
    mn, mx = df_fe[col].min(), df_fe[col].max()
    if mx > mn:
        df_fe[col + '_norm'] = (df_fe[col] - mn) / (mx - mn)

norm = lambda c: c + '_norm' if c + '_norm' in df_fe.columns else c

df_fe['Career_Readiness_Score'] = (
    0.30 * df_fe[norm('CGPA')]
    + 0.20 * df_fe[norm('Coding_Skills')]
    + 0.15 * df_fe[norm('Communication_Skills')]
    + 0.15 * df_fe[norm('Aptitude_Test_Score')]
    + 0.10 * df_fe[norm('Internships')]
    + 0.10 * df_fe[norm('Certifications')]
) if all(c in df.columns for c in ['CGPA','Coding_Skills','Communication_Skills',
                                    'Aptitude_Test_Score','Internships','Certifications']) else None

if df_fe['Career_Readiness_Score'] is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].hist(df_fe['Career_Readiness_Score'], bins=30, color='purple', alpha=0.7, edgecolor='white')
    axes[0].set_title('Career Readiness Score Distribution')

    sns.boxplot(data=df_fe, x=target_col, y='Career_Readiness_Score',
                palette=['#e74c3c', '#2ecc71'], ax=axes[1])
    axes[1].set_title('Career Readiness Score vs Placement')
    plt.tight_layout()
    plt.show()

    print('='*55)
print('           EDA SUMMARY')
print('='*55)
print(f'Total Records      : {len(df)}')
print(f'Total Features     : {df.shape[1]}')
print(f'Numerical Features : {len(num_cols)}')
print(f'Categorical Features: {len(cat_cols)}')
print(f'Missing Values     : {df.isnull().sum().sum()}')
print(f'\nPlacement Distribution:')
print(df[target_col].value_counts(normalize=True).mul(100).round(2).to_string())
print('\nTop 3 features correlated with placement:')
print(target_corr.abs().sort_values(ascending=False).head(3))
print('='*55)
